# Unified Notebook 
* This notebook will act as a gateway to generate the diagrams used in the report
* To train the TPCRPs individually, please refer to the TPCRP_Algorithms directory, and execute the .py's individually, they will train for 500 epochs, though this can be reduced from the __init__ method
* **Please note** the majority of graphs are already present in the "Evaluation Metrics" Directory 

* Thus the training pipeline will be found in TPCRP_Algorithm

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt  
import seaborn as sns  
import random 

import torch 

import torch.nn as nn 
from torch.utils.data import DataLoader, Subset


import torchvision 
import torchvision.transforms as transforms 

from sklearn.datasets import make_blobs
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

from typing import List, Tuple

from sklearn.decomposition import PCA 


from Notebooks.supervised_training import train_supervised

In [ ]:
budgets = [10,20,40,80]
seeds = [0,1,2,3,4]

# Unsupervised
typiclust_unsup = {
    B: np.load(f"budget_results/unsupervised_budget_results/typiclust_B{B}.npy")
    for B in budgets
}

# Self-supervised (TPCRP)
typiclust_ssl = {
    B: np.load(f"budget_results/unmodified_budget_results/typiclust_B{B}.npy")
    for B in budgets
    
}

modified_typiclust = {
    B : np.load(f"budget_results/modified_budget_results/typiclust_B{B}.npy")
    for B in budgets
}

# Fully supervised
typiclust_sup = {
    B: np.load(f"budget_results/supervised_budget_results/typiclust_B{B}.npy")
    for B in budgets
}


# 'Features' from SSL Unmodified ( self-supervised ) - not possible to send via GitHub due to large size
# 'Features' can be independently ( and reliably ) obtained by running the TPCRP_Algorithm/Modified_TPCRP_Algorithm.py
#features = np.load("budget_results/modified_budget_results/features.npy")

# Cross-Framework-Comparison - for comparison between the different frameworks used 

In [ ]:
N = 50000 # CIFAR-10 SIZE
def generate_random_indices(seed, B):
    rng = np.random.RandomState(seed)
    
    return rng.choice(N, size=B, replace=False)

# Please note : The cell below will take some time to complete 

In [ ]:
results = []

for seed in seeds:
    # Recreational
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


    for B in budgets:

        # --- Unsupervised TypiClust ---
        idx = typiclust_unsup[B]
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Unsupervised", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

        # --- Self-supervised TypiClust ---
        idx = typiclust_ssl[B]
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Self-supervised", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

        # --- Fully supervised TypiClust ---
        idx = typiclust_sup[B]
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Supervised", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

        # --- Random baseline ---
        idx = generate_random_indices(seed, B)
        acc, loss, runtime = train_supervised(idx, epochs=50)
        results.append({"method": "Random", "budget": B, "seed": seed,
                        "accuracy": acc, "runtime": runtime})

df = pd.DataFrame(results)
df.to_csv("typiclust_framework_comparison.csv", index=False)
df


In [ ]:
plt.figure(figsize=(8,6))
sns.lineplot(data=df, x="budget", y="accuracy", hue="method", marker="o")
plt.title("Accuracy vs Budget Across Frameworks")
plt.grid(True)
plt.show()


In [ ]:
df_random = df[df["method"] == "Random"].set_index(["budget", "seed"])

df["acc_diff"] = df.apply(
    lambda row: row["accuracy"] - df_random.loc[(row["budget"], row["seed"]), "accuracy"],
    axis=1
)

plt.figure(figsize=(8,6))

sns.lineplot(data=df[df["method"] != "Random"],
             x="budget", y="acc_diff", hue="method", marker="o")
plt.axhline(0, color="black", linestyle="--")
plt.title("Accuracy Difference vs Random Baseline")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(typiclust_unsup[10], label="Unsupervised", kde=True)
sns.histplot(typiclust_ssl[10], label="Self-supervised", kde=True)
sns.histplot(typiclust_sup[10], label="Supervised", kde=True)
plt.legend()
plt.title("Distribution of Selected Indices (B=10)")
plt.show()


# HDBSCAN Notebook - for the modified Algorithm
* Please note, this section is commented out to prevent errors 
* Note : If you wish to test this system out, please run TPCRP_Algorithm/Modified_TPCRP_Algorithm.py to get the feature matrix, and place it in the budget_results/modified_budget_results directory. Thank you!

In [ ]:
"""
pca = PCA(n_components=2) # 2D
proj = pca.fit_transform(features) # Disabled due to features not being able to be committed ( GitHub storage restrictions )
"""

In [ ]:
"""
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    ),
])

dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
"""

In [ ]:
"""
def plot_typiclust_visuals(indices, proj, dataset, B, title_prefix):
    limit = 20

    # -----------------------------
    # 1. PCA manifold plot
    # -----------------------------
    plt.figure(figsize=(7,7))
    plt.scatter(proj[:,0], proj[:,1], s=5, alpha=0.3, label="All points")
    plt.scatter(proj[indices,0], proj[indices,1], s=40, color="red", label="Selected")
    plt.title(f"{title_prefix} — PCA Manifold (B={B})")
    plt.legend()
    plt.show()

    # -----------------------------
    # 2. Index scatter
    # -----------------------------
    plt.figure(figsize=(10,2))
    plt.scatter(indices, [0]*len(indices), s=30)
    plt.title(f"{title_prefix} — Selected Indices (B={B})")
    plt.yticks([])
    plt.xlabel("Dataset Index")
    plt.show()

    # -----------------------------
    # 3. CIFAR‑10 thumbnails
    # -----------------------------
    plt.figure(figsize=(12,6))
    for i, idx in enumerate(indices[:limit]):
        img, label = dataset[idx]
        img = img.permute(1,2,0).numpy()
        img = (img * 0.2470 + 0.4914).clip(0,1)

        plt.subplot(4,5,i+1)
        plt.imshow(img)
        plt.title(f"{label}")
        plt.axis("off")

    plt.suptitle(f"{title_prefix} — First {limit} Selected Samples (B={B})")
    plt.show()
"""

In [ ]:
"""
for B in budgets:
    print(f"\n=/\ BUDGET {B} /\=")

    plot_typiclust_visuals(typiclust_unsup[B], proj, dataset, B, "Unsupervised TypiClust")
    plot_typiclust_visuals(typiclust_ssl[B], proj, dataset, B, "Self-Supervised TypiClust")
    plot_typiclust_visuals(typiclust_sup[B], proj, dataset, B, "Supervised TypiClust")
    #plot_typiclust_visuals(modified_typiclust[B], proj, dataset, B, "Modified TypiClust")

    # PLEASE NOTE : the 'features' accessed to be used cannot be saved on the GitHub Repository due to a size > 100MB ( Added to .gitignore )
    # The Modified TypiClust consequently cannot be accessed - so I have commented it out 

"""




# Uncertainity-Baseline Plot Design 

In [ ]:
def get_initial_seed(train_dataset, n_per_class=1):
    labels = np.array(train_dataset.targets)
    seed = []
    
    for c in range(10): 
        idx = np.where(labels == c)[0]
        seed.append(int(np.random.choice(idx)))
    return seed

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),

        (0.2470, 0.2435, 0.2616)
    ),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=base_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=base_transform
)

test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
budgets = [10, 20, 40, 80]

class SimpleClassifier(nn.Module):
    def __init__(self, dropout=False):
        super().__init__()

        self.dropout = dropout

        self.features = torchvision.models.resnet18(pretrained=False)
        
        self.features.fc = nn.Identity()  # remove final layer

        self.classifier = nn.Sequential(
            nn.Dropout(0.5) if dropout else nn.Identity(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat)


In [ ]:
# Various method calculation functions 

def score_least_confidence(logits):

    probs = torch.softmax(logits ,  dim=1)
    return ( 1- probs.max(dim=1).values)

def score_margin(logits):
    probs = torch.softmax(logits,dim=1)
    top2 = torch.topk(probs, 2, dim=1).values

    return top2[:,0] - top2[:,1] 

def score_entropy(logits):
    probs = torch.softmax(logits,dim=1)
    return - 1 * ( probs * probs.log()).sum(dim=1)

def compute_badge_embeddings(model, loader, device):
    model.eval()
    grads = [] 




    for x, _ in loader:
        # Device implemented
        x = x.to(device)
        x.requires_grad=True

        logits =model(x)
        probs = torch.softmax(logits, dim=1)

        y_hat = probs.argmax(dim=1)



        loss = F.cross_entropy(logits, y_hat, reduction='sum')
        loss.backward()


        grads.append(x.grad.view(x.size(0), -1).cpu().numpy())
    return np.concatenate(grads, axis=0)

def select_by_uncertainty(model, unlabeled_loader, device, B, scoring_fn):
    model.eval()

    scores = []


    with torch.no_grad():
        
        for x, idx in unlabeled_loader:
            x=x.to(device)
            logits = model(x)
            s =  scoring_fn(logits)
            scores.extend(list(zip(idx.numpy(), s.cpu(). numpy()) ))

    # desc. uncertainity
    scores = sorted(scores, key=lambda x: -x[1])
    selected = [idx for idx, _ in scores[:B]]

    return selected 



In [ ]:
def plot_baseline_comparison(results_dict, budgets):
    plt.figure(figsize=(8,5))
    
    for method, accs in results_dict.items():
        plt.plot(budgets, accs, marker="x", label=method)

    plt.xlabel("Budget")
    plt.ylabel("Accuracy")

    plt.title("Active Learning baseline comparison")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

Loop for Active Learning Below

In [ ]:
def evaluate(model, loader, device): 
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        
        for x, y in loader:
            x,y = x.to(device), y.to(device)
            logits = model(x)

            preds = logits.argmax(dim=1)
            correct += (preds ==y ).sum().item()

            total += y.size(0)
        
        return correct / total 

In [ ]:
def active_learning_round(
    model,
    train_dataset,
    test_loader,

    labeled_indices,
    unlabeled_indices,
    B,
    scoring_fn,
    device
):
    labeled_subset = torch.utils.data.Subset(train_dataset, labeled_indices)
    loader = DataLoader(labeled_subset, batch_size=128, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(5):  
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    unlabeled_subset = torch.utils.data.Subset(train_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

    selected = select_by_uncertainty(model, unlabeled_loader, device, B, scoring_fn)
    selected_global = [unlabeled_indices[i] for i in selected]

    new_labeled = labeled_indices + selected_global
    new_unlabeled = [i for i in unlabeled_indices if i not in selected_global]

    acc = evaluate(model, test_loader, device)

    return new_labeled, new_unlabeled, acc


In [ ]:
def mc_dropout_predict(model, x, T=3):
    model.train()  # keep dropout ON
    preds = []
    for _ in range(T):
        logits = model(x)
        preds.append(torch.softmax(logits, dim=1).unsqueeze(0))
    return torch.cat(preds, dim=0)  # [t, b,   c]

# BALD method 
def score_bald_from_mc(mc_probs):
    
    mean_probs = mc_probs.mean(dim=0)
    entropy_mean = -(mean_probs * mean_probs.log()).sum(dim=1)
    entropy_expected = -(mc_probs * mc_probs.log()).sum(dim=2).mean(dim=0)
    return entropy_mean - entropy_expected


def score_bald(model, x, device, T=3):
    x = x.to(device)
    mc_probs = mc_dropout_predict(model, x, T=T)
    return score_bald_from_mc(mc_probs)

def score_dbal_from_mc(mc_probs):
    preds = mc_probs.argmax(dim=2)  # [T, B]
    mode_counts = torch.mode(preds, dim=0).values
    return 1 - (mode_counts / mc_probs.size(0))


In [ ]:
def select_badge(model, unlabeled_loader, device, B):
    model.eval()
    embeddings = compute_badge_embeddings(model, unlabeled_loader, device)

    # kmeans method 
    kmeans = KMeans(n_clusters=B, init='k-means++')
    kmeans.fit(embeddings)

    centers = kmeans.cluster_centers_
    dists = np.linalg.norm(embeddings[:, None] - centers[None, :], axis=2)
    selected = np.argmin(dists, axis=0)

    all_indices = []
    for _, idx in unlabeled_loader:
        
        all_indices.extend(idx.numpy())

    return [all_indices[i] for i in selected]


In [ ]:
def select_random(unlabeled_indices, B):
    return list(np.random.choice(unlabeled_indices, size=B, replace=False))


In [ ]:
def evaluate_fixed_selection(model, train_dataset, test_loader, indices, device):
    subset = Subset(train_dataset, indices)
    loader = DataLoader(subset, batch_size=128, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()


    model.train()
    for epoch in range(5):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return evaluate(model, test_loader, device)


In [ ]:
def run_all_baselines(train_dataset, test_loader, budgets, device):
    print("\n=== Starting all baselines ===")

    results = {
        "Least Confidence": [],
        "Margin": [],
        "Entropy": [],
        "BADGE": [],
        "BALD": [],
        "DBAL": [],
        "Random": [],
        "TPC-RP": [], 
        "TPC-DC": []  
    }

    scoring_fns = {
        "Least Confidence": score_least_confidence,
        "Margin": score_margin,
        "Entropy": score_entropy

    }


    for method, fn in scoring_fns.items():
        print(f"\n=== Running baseline: {method} ===")
        labeled = get_initial_seed(train_dataset)
        unlabeled = list(range(len(train_dataset)))
        accs = []

        for B in budgets:
            print(f"  -> Budget {B}: training classifier...")
            model = SimpleClassifier().to(device)

            labeled, unlabeled, acc = active_learning_round(
                model, train_dataset, test_loader,
                labeled, unlabeled, B,
                fn, device
            )

            print(f"     Accuracy after budget {B}: {acc:.4f}")
            accs.append(acc)

        results[method] = accs


    print("\n=== Running baseline: BADGE ===")
    labeled = get_initial_seed(train_dataset)

    unlabeled = list(range(len(train_dataset)))
    accs = []

    for B in budgets:
        print(f"  -> BADGE selecting {B} points...")
        model = SimpleClassifier().to(device)

        unlabeled_subset = Subset(train_dataset, unlabeled)
        unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

        selected = select_badge(model, unlabeled_loader, device, B)
        labeled += selected
        unlabeled = [i for i in unlabeled if i not in selected]

        acc = evaluate(model, test_loader, device)
        print(f"     Accuracy after BADGE budget {B}: {acc:.4f}")
        accs.append(acc)

    results["BADGE"] = accs


    print("\n=== Running baseline: BALD ===")
    labeled = get_initial_seed(train_dataset)
    unlabeled = list(range(len(train_dataset)))
    accs = []

    for B in budgets:
        print(f"  -> BALD scoring unlabeled pool for budget {B}...")
        model = SimpleClassifier(dropout=True).to(device)

        unlabeled_subset = Subset(train_dataset, unlabeled)
        unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

        selected = []
        for x, idx in unlabeled_loader:
            scores = score_bald(model, x, device)
            selected.extend(list(zip(idx.numpy(), scores.cpu().numpy())))

        selected = sorted(selected, key=lambda x: -x[1])[:B]
        selected = [i for i, _ in selected]

        labeled += selected
        unlabeled = [i for i in unlabeled if i not in selected]

        acc = evaluate(model, test_loader, device)
        print(f"     Accuracy after BALD budget {B}: {acc:.4f}")
        accs.append(acc)

    results["BALD"] = accs



    print("\n=== Running baseline: DBAL ===")
    
    labeled = get_initial_seed(train_dataset)
    unlabeled = list(range(len(train_dataset)))
    accs = []

    for B in budgets:
        print(f"  -> DBAL scoring unlabeled pool for budget {B}...")
        model = SimpleClassifier(dropout=True).to(device)

        unlabeled_subset = Subset(train_dataset, unlabeled)
        unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

        selected = []
        for x, idx in unlabeled_loader:
            mc_probs = mc_dropout_predict(model, x.to(device))
            scores = score_dbal_from_mc(mc_probs)
            selected.extend(list(zip(idx.numpy(), scores.cpu().numpy())))

        selected = sorted(selected, key=lambda x: -x[1])[:B]
        selected = [i for i, _ in selected]

        labeled += selected
        unlabeled = [i for i in unlabeled if i not in selected]

        acc = evaluate(model, test_loader, device)
        print(f"     Accuracy after DBAL budget {B}: {acc:.4f}")
        accs.append(acc)

    results["DBAL"] = accs

    print("\n=== Evaluating TPC-RP selections ===")
    for B in budgets:
        print(f"  -> Evaluating TPC-RP for budget {B}...")
        indices = np.load(f"typiclust_B{B}.npy")
        model = SimpleClassifier().to(device)
        acc = evaluate_fixed_selection(model, train_dataset, test_loader, indices, device)
        print(f"     TPC-RP accuracy for budget {B}: {acc:.4f}")
        results["TPC-RP"].append(acc)
    print("\n=== Evaluating TPC-DC selections ===")
    for B in budgets:
        print(f"  -> Evaluating TPC-DC for budget {B}...")
        try:
            indices = np.load(f"tpcdc_B{B}.npy")
            model = SimpleClassifier().to(device)
            acc = evaluate_fixed_selection(model, train_dataset, test_loader, indices, device)
            print(f"     TPC-DC accuracy for budget {B}: {acc:.4f}")
            results["TPC-DC"].append(acc)
        except FileNotFoundError:
            print(f"     No TPC-DC file found for budget {B}.")
            results["TPC-DC"].append(None)

    print("\n=== All baselines complete ===")
    return results


In [ ]:
results = run_all_baselines(train_dataset, test_loader, budgets, device)
plot_baseline_comparison(results, budgets)

# This will download the files ( plots ) 
